# Region-Specific Analysis

Analyze per-region dataset coverage, readiness, and embedding behavior
across governed datasets.

> **Warning:** Results are diagnostic only. True region-specific conclusions
> require real microscope/gold-set capture data. Current datasets (smoke, lab,
> public external) provide structural validation only.

## Curated Results

See [Region-Specific Analysis Results](../docs/datasets/region-specific-analysis-results.md)
for the full curated summary of FabricNET and LeaData region analysis.

## Run Analysis

From `auth-platform/`:

```bash
# Smoke dataset
make region-analysis-smoke

# FabricNET (external regions included)
make region-analysis-fabricnet

# LeaData (external regions included)
make region-analysis-leadata

# Compare all
make compare-region-analysis
```

In [ ]:
import json
from pathlib import Path

%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

# Configuration — change as needed
DATASET = "external_fabricnet_v0.1.0"

ANALYSIS_DIR = Path("../experiments/outputs/region-analysis")
ds_tag = DATASET.replace(".", "_")
analysis_path = ANALYSIS_DIR / f"{ds_tag}__region_analysis.json"

print(f"Looking for: {analysis_path}")
print(f"Exists: {analysis_path.exists()}")

In [ ]:
# Load analysis
if not analysis_path.exists():
    print("No analysis file found. Run the analysis first:")
    print(f"  make region-analysis-fabricnet")
    data = None
else:
    data = json.loads(analysis_path.read_text())
    print(f"Dataset:    {data['dataset_version']}")
    print(f"Model:      {data['model_version']}")
    print(f"Records:    {data['total_records']}")
    print(f"Embeddings: {data['total_embeddings']}")
    print()

    # Coverage table
    header = f"{'Region':<30} {'Images':>6} {'Items':>6} {'Readiness':<20} {'Warnings'}"
    print(header)
    print("-" * len(header))
    for c in data["coverage"]:
        warns = ", ".join(c.get("warnings", [])) or ""
        print(f"{c['region_code']:<30} {c['image_count']:>6} {c['item_count']:>6} "
              f"{c['readiness']:<20} {warns}")

In [ ]:
# Readiness summary
if data:
    readiness_counts = {}
    for c in data["coverage"]:
        r = c["readiness"]
        readiness_counts[r] = readiness_counts.get(r, 0) + 1
    print("Readiness distribution:")
    for level in ["no_data", "sparse", "diagnostic_only", "evaluation_ready", "training_candidate"]:
        count = readiness_counts.get(level, 0)
        if count:
            print(f"  {level}: {count} region(s)")

In [ ]:
# Bar chart: images per region
if data:
    regions = [c["region_code"] for c in data["coverage"]]
    counts = [c["image_count"] for c in data["coverage"]]

    fig, ax = plt.subplots(figsize=(10, max(3, len(regions) * 0.6)))
    bars = ax.barh(regions, counts, color="steelblue")
    ax.set_xlabel("Image Count")
    ax.set_title(f"Images per Region — {DATASET}")
    for bar, val in zip(bars, counts):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                str(val), va="center", fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print("No data to plot.")

In [ ]:
# Bar chart: cluster separation by region (if metrics available)
if data:
    rm_with_metrics = [rm for rm in data.get("region_metrics", []) if rm.get("metrics")]
    if rm_with_metrics:
        rm_regions = [rm["region_code"] for rm in rm_with_metrics]
        rm_cs = [rm["metrics"]["cluster_separation"] for rm in rm_with_metrics]

        fig, ax = plt.subplots(figsize=(10, max(3, len(rm_regions) * 0.6)))
        bars = ax.barh(rm_regions, rm_cs, color="darkorange")
        ax.set_xlabel("Cluster Separation")
        ax.set_title(f"Cluster Separation by Region — {DATASET}")
        for bar, val in zip(bars, rm_cs):
            ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height() / 2,
                    f"{val:.4f}", va="center", fontsize=9)
        plt.tight_layout()
        plt.show()
    else:
        print("No per-region metrics with data available.")

In [ ]:
# Load additional datasets for comparison (optional)
for other_ds in ["external_leadata_v0.1.0", "smoke_lab_lv_monogram_v0.1.0"]:
    other_tag = other_ds.replace(".", "_")
    other_path = ANALYSIS_DIR / f"{other_tag}__region_analysis.json"
    if other_path.exists():
        other = json.loads(other_path.read_text())
        print(f"\n{other['dataset_version']}:")
        for c in other["coverage"]:
            print(f"  {c['region_code']:<30} images={c['image_count']:>5}  readiness={c['readiness']}")
    else:
        print(f"No analysis for {other_ds}")

## Notes / Conclusions

- Current datasets are structural — true region-specific conclusions require real
  microscope/gold-set capture data with multiple regions per item.
- Public external datasets (FabricNET, LeaData) use single EXTERNAL_* region codes,
  so per-region comparison within those datasets is limited.
- Smoke/lab datasets may have multiple luxury-item region codes but small sample sizes.
- Region readiness scores are based on image count only; label quality, item
  diversity, and provenance also matter.
- This framework is ready to be applied to real capture data when available.

**These results are diagnostic only.** They do not prove authentication accuracy.